In [5]:
import numpy as np
import os
import pandas as pd
import json
import pickle
import openai
import json_repair
import copy as cp

#from blablador import Models, Completions, ChatCompletions, TokenCount
import ast
import re
from itertools import chain
#import country_converter as coco
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy
import time
from openai import OpenAI
from pathlib import Path

from src.data import *
from src.impact_def import *
from src.prompts_impacts import *
from src.LLM_functions import *
from src.classOutput import *
from src.post_process_functions import *

In [ ]:
#prepare test data
test_dicts = [
    {"text": "Over 5,000 people have been impacted by blocked roads, power outages, and severe shortages of food, water, and essential supplies.",
     "appealCode" : None,
     "reportDate": None
    },
    {"text": "100 people lost access to healthcare.",
     "appealCode" : None,
     "reportDate": None
    },
    {"text": "More of 100m USD (110m CHF) of damage to infrastructure has been reported.",
     "appealCode" : None,
     "reportDate": None
    },

]
test_df = pd.DataFrame(test_dicts)

In [7]:
test_df

,text,appealCode,reportDate
0,"Over 5,000 people have been impacted by blocke...",None,None
1,100 people lost access to healthcare.,None,None
2,More of 100m USD (110m CHF) of damage to infra...,None,None


In [8]:
#try new formulation with formatting
def quantify_impacts_type_value_loc_date_haz_const_unit(text, imp_main, imp_sub, imp_unit, hazard_cat):
    """Find impact type, values, locs, etc. altogether"""
    prompt = f"""
    Context information is below.
    ---
    {text}
    ---
    Using information from the text above and no previous knowledge, please answer the query.
    Query: Extract from the above text all descriptions and mentions of impacts resulting from extreme
    natural hazard events. Answer by providing a list of JSONs, following strictly the instructructions
    on the field to extracts and the structure of the output below:
    [
    {{
        "impactType": "<one of {imp_main}>",  # Description: Main type of impact (e.g., Human, Agriculture, Infrastructure).
        "impactSubtype": "<one of {imp_sub}>",  # Description: Subtype of impact (e.g., Affected People, Crop Production, WASH Infrastructure).
        "impactValue": <integer or null>,  # Description: The quantified value of the impact. Use null if unknown.
        "impactUnit": "<one of {imp_unit}>",  # Description: The unit of the impact value (e.g., people, kilometers, houses).
        "impactValueFlag": "one of ["exact", "approx"], # Description: The quality flag for the quantified impact value informing on the confidence in the quantified value.
        "country": "<string>",  # Description: The country where the impact occurred.
        "location": ["<list of strings>" or null],  # Description: A list of affected locations (e.g., cities, regions).
        "startYear": <integer or null>,  # Description: The year when the impact started. Use null if unknown.
        "startMonth": <integer or null>,  # Description: The month when the impact started. Use null if unknown.
        "startDay": <integer or null>,  # Description: The day when the impact started. Use null if unknown.
        "endYear": <integer or null>,  # Description: The year when the impact ended. Use null if ongoing or unknown.
        "endMonth": <integer or null>,  # Description: The month when the impact ended. Use null if ongoing or unknown.
        "endDay": <integer or null>,  # Description: The day when the impact ended. Use null if ongoing or unknown.
        "hazards": ["<one or more of {hazard_cat}>" or null],  # Description: List of hazards causing the impact (e.g., Tropical storm, Drought, Flood).
        "impactsAnnotation": ["<list of strings>"]  # Description: the exact text excerpt from where you extracted the impacts information. Write the text exactly as found in the original text.
    }}
    ]

    Rules:
    1. The `impactType` field must only contain one of the valid values: {imp_main}.
    2. The `impactSubtype` field must only contain one of the valid values: {imp_sub}.
    2. The `impactUnit` field must only contain one of the valid values: {imp_unit}.
    3. The `hazards` field must only contain values from {hazard_cat}.
    4. Follow the JSON format strictly; do not add or remove fields.
    5. Ensure all field constraints are respected. If any values are unknown, use null.
    6. Do not add notes or extra text, only output the list of JSONs.
    7. Do not reuse a specific impactValue for different entries.

    Examples:\n""" + \
    examples
    return prompt

def quantify_impacts_type_value_loc_date_haz_free_unit(text, imp_main, imp_sub, imp_unit, imp_unittype, hazard_cat):
    """Find impact type, values, locs, etc. altogether"""
    prompt = f"""
    Context information is below.
    ---
    {text}
    ---
    Using information from the text above and no previous knowledge, please answer the query.
    Query: Extract from the above text all descriptions and mentions of impacts resulting from extreme
    natural hazard events. Answer by providing a list of JSONs, following strictly the instructructions
    on the field to extracts and the structure of the output below:
    [
    {{
        "impactSubtype": "<one of {imp_sub}>",  # Description: Subtype of impact (e.g., Affected People, Crop Production, WASH Infrastructure).
        "impactValue": "<string>" or null,  # Description: The quantified value of the impact. Use the exact number if mentioned, or retain the text or range as provided for vague numbers. Use null if unknown.
        "impactValuePrecision": "one of ["exact", "approx"], # Description: The quality flag for the quantified impact value informing on the precision in the quantified value.
        "impactUnit": "<string>" or null,  # Description: The unit of the impact value (e.g., people, meters, houses). Use null if unknown.
        "country": "<string>",  # Description: The country where the impact occurred.
        "location": ["<list of strings>" or null],  # Description: A list of affected locations (e.g., cities, regions).
        "startYear": <integer or null>,  # Description: The year when the impact started. Use null if unknown.
        "startMonth": <integer or null>,  # Description: The month when the impact started. Use null if unknown.
        "startDay": <integer or null>,  # Description: The day when the impact started. Use null if unknown.
        "endYear": <integer or null>,  # Description: The year when the impact ended. Use null if ongoing or unknown.
        "endMonth": <integer or null>,  # Description: The month when the impact ended. Use null if ongoing or unknown.
        "endDay": <integer or null>,  # Description: The day when the impact ended. Use null if ongoing or unknown.
        "hazards": ["<one or more of {hazard_cat}>" or null],  # Description: List of hazards causing the impact (e.g., Tropical storm, Drought, Flood).
        "impactsAnnotation": ["<list of strings>"]  # Description: the exact text excerpt from where you extracted the impacts information. Write the text exactly as found in the original text.
    }}
    ]

    Rules:
    - The `impactType` field must only contain one of the valid values: {imp_main}.
    - The `hazards` field must only contain values from {hazard_cat}.
    - Do not reuse a specific impactValue for different entries.
    - Follow the JSON format strictly; do not add or remove fields.
    - Ensure all field constraints are respected. If any values are unknown, use null.
    - Do not add notes or extra text, only output the list of JSONs.

    Examples:\n""" + examples
    return prompt

def quantify_impacts_type_value_loc_date_haz_free_unit_ranges(text, imp_main, imp_sub, imp_unit, imp_unittype, hazard_cat):
    """Find impact type, values, locs, etc. altogether"""
    prompt = f"""
    Context information is below.
    ---
    {text}
    ---
    Using information from the text above and no previous knowledge, please answer the query.
    Query: Extract from the above text all descriptions and mentions of impacts resulting from extreme
    natural hazard events. Answer by providing a list of JSONs, following strictly the instructructions
    on the field to extracts and the structure of the output below:
    [
    {{
        "impactSubtype": "<one of {imp_sub}>",  # Description: Subtype of impact (e.g., Affected People, Crop Production, WASH Infrastructure).
        "impactValue": <float or null>  # Description: The quantified value of the impact. Provide the exact number if mentioned. If a range is provided give the upper estimate. Use null if unknown.
        "impactValueApprox": "one of ["exact", "approx"], # Description: The flag describing whether the quantified impact value is exact or approximate. Use null if unknown.
        "impactValueMin": <float or null>,  # Description: The lower bound estimate of the quantified value of the impact if the impact is approximate or a range. Use null if unknown.
        "impactValueMax": <float or null>,  # Description: The upper bound estimate of the quantified value of the impact if the impact is approximate or a range. Use null if unknown.
        "impactUnit": "<string>" or null,  # Description: The unit of the impact value (e.g., people, meters, houses). Use null if unknown.
        "country": ["<list of strings>" or null],  # Description: The list of affected countries where the described impact occurred.
        "location": ["<list of strings>" or null],  # Description: The list of affected locations (e.g., cities, regions) where the described impact occurred.
        "startYear": <integer or null>,  # Description: The year when the impact started. Use null if unknown.
        "startMonth": <integer or null>,  # Description: The month when the impact started. Use null if unknown.
        "startDay": <integer or null>,  # Description: The day when the impact started. Use null if unknown.
        "endYear": <integer or null>,  # Description: The year when the impact ended. Use null if ongoing or unknown.
        "endMonth": <integer or null>,  # Description: The month when the impact ended. Use null if ongoing or unknown.
        "endDay": <integer or null>,  # Description: The day when the impact ended. Use null if ongoing or unknown.
        "hazards": ["<one or more of {hazard_cat}>" or null],  # Description: List of hazards causing the impact (e.g., Tropical storm, Drought, Flood).
        "impactsAnnotation": ["<list of strings>"]  # Description: the exact text excerpt from where you extracted the impacts information. Write the text exactly as found in the original text.
    }}
    ]

    Rules:
    - The `impactType` field must only contain one of the valid values: {imp_main}.
    - The `hazards` field must only contain values from {hazard_cat}.
    - Do not reuse a specific impactValue for different entries.
    - Provide the unit of the impact in the `impactUnit` field exactly as found in the original text, keeping all information on the measured quantity (e.g. write 'km of roads' instead of just 'km')
    - When the `impactValue` field is an exact number, `impactValueApprox` must be set to `exact`.
    - When the `impactValue` field is an approximation, `impactValueApprox` must be set to `approx`.
    - When the `impactValue` field is a range, `impactValueApprox` must be set to `approx` and the `impactValueMin` and `impactValueMax` fields must be filled.
    - Follow the JSON format strictly; do not add or remove fields.
    - Ensure all field constraints on allowed values and their data types are respected.
    - Do not add notes or extra text, only output the list of JSONs.

    Examples:\n""" + examples_range
    return prompt

def quantify_impacts_type_value_loc_date_haz_free_unit_type_subtype(text, imp_main, imp_sub, imp_unit, imp_unittype, hazard_cat):
    """Find impact type, values, locs, etc. altogether"""
    prompt = f"""
    Context information is below.
    ---
    {text}
    ---
    Using information from the text above and no previous knowledge, please answer the query.
    Query: Extract from the above text all descriptions and mentions of impacts resulting from extreme
    natural hazard events. Answer by providing a list of JSONs, following strictly the instructructions
    on the field to extracts and the structure of the output below:
    [
    {{
        "impactType": "<one of {imp_main}>",  # Description: Main type of impact (e.g., Human, Agriculture, Infrastructure).
        "impactSubtype": "<one of {imp_sub}>",  # Description: Subtype of impact (e.g., Affected People, Crop Production, WASH Infrastructure).
        "impactValue": "<string>" or null,  # Description: The quantified value of the impact. Use the exact number if mentioned, or retain the text or range as provided for vague numbers. Use null if unknown.
        "impactValueApprox": "<boolean>", # Description: Boolean indicating if the quantified value is an approximation or a range.
        "impactValuePrecision": "one of ["exact", "approx"], # Description: The quality flag for the quantified impact value informing on the precision in the quantified value. Use null if unknown.
        "impactUnit": "<string>" or null,  # Description: The unit of the impact value (e.g., people, meters, houses). Use null if unknown.
        "country": "<string>",  # Description: The country where the impact occurred.
        "location": ["<list of strings>" or null],  # Description: A list of affected locations (e.g., cities, regions).
        "startYear": <integer or null>,  # Description: The year when the impact started. Use null if unknown.
        "startMonth": <integer or null>,  # Description: The month when the impact started. Use null if unknown.
        "startDay": <integer or null>,  # Description: The day when the impact started. Use null if unknown.
        "endYear": <integer or null>,  # Description: The year when the impact ended. Use null if ongoing or unknown.
        "endMonth": <integer or null>,  # Description: The month when the impact ended. Use null if ongoing or unknown.
        "endDay": <integer or null>,  # Description: The day when the impact ended. Use null if ongoing or unknown.
        "hazards": ["<one or more of {hazard_cat}>" or null],  # Description: List of hazards causing the impact (e.g., Tropical storm, Drought, Flood).
        "impactsAnnotation": ["<list of strings>"]  # Description: the exact text excerpt from where you extracted the impacts information. Write the text exactly as found in the original text.
    }}
    ]

    Rules:
    - The `impactType` field must only contain one of the valid values: {imp_main}.
    - The `hazards` field must only contain values from {hazard_cat}.
    - Do not reuse a specific impactValue for different entries.
    - Follow the JSON format strictly; do not add or remove fields.
    - Ensure all field constraints are respected. If any values are unknown, use null.
    - Do not add notes or extra text, only output the list of JSONs.

    Examples:\n""" + examples_type_subtype
    return prompt

examples = """[{
       "impactSubtype" : "Affected People",
       "impactValue": "10000",
       "impactUnit": "people",
       "impactValuePrecision" : "exact",
       "country" : "Sudan",
       "location" : ["Abu Hamad", "Tokar"],
       "startYear" :"2024,
       "startMonth" :"08,
       "startDay" : 29,
       "endYear" : "null",
       "endMonth" : "null",
       "endDay" : "null",
       "hazards" : ["Mass Movement"],
       "impactsAnnotation" : ["Landslides impacted 10000 people in the cities of Abu Hamad and Tokar on the 29 August 2024",]
      },
      {
       "impactSubtype" : "Crop Production and Forestry",
       "impactValue": "100 to 200",
       "impactUnit": "kg of crop production",
       "impactValuePrecision" : "approx",
       "country" : "Sudan",
       "location" : ["Red Sea State"],
       "startYear" : 2024,
       "startMonth" : 08,
       "startDay" : "null",
       "endYear" : 2024,
       "endMonth" : 10,
       "endDay" : "null",
       "hazards" : ["Flood"],
       "impactsAnnotation" : ["Flash floods impacted 100 to 200 kg of crop production in Red Sea State between August to October 2024"]
      },
      {
       "impactSubtype" : "Healthcare Infrastructure",
       "impactValue": "4",
       "impactUnit": "healthcare facilities",
       "impactValueFlag" : "approx",
       "country" : "Sudan",
       "location": ["River Nile State"],
       "startYear": "null",
       "startMonth": "null",
       "startDay": "null",
       "endYear": "null",
       "endMonth": "null",
       "endDay": "null",
       "hazards" : ["Convective Storm],
       "impactsAnnotation" : ["At least 4 hospitals have been impacted by a hailstorm in River Nile State alone."]
       }]
    """
examples_type_subtype = """[{
       "impactType" : "Human",
       "impactSubtype" : "Affected People",
       "impactValue": "10000",
       "impactUnit": "people",
       "impactValuePrecision" : "exact",
       "country" : "Sudan",
       "location" : ["Abu Hamad", "Tokar"],
       "startYear" :"2024,
       "startMonth" :"08,
       "startDay" : 29,
       "endYear" : "null",
       "endMonth" : "null",
       "endDay" : "null",
       "hazards" : ["Mass Movement"],
       "impactsAnnotation" : ["Landslides impacted 10000 people in the cities of Abu Hamad and Tokar on the 29 August 2024",]
      },
      {
       "impactType" : "Agriculture",
       "impactSubtype" : "Crop Production and Forestry",
       "impactValue": "100 to 200",
       "impactUnit": "kg of crop production",
       "impactValuePrecision" : "approx",
       "country" : "Sudan",
       "location" : ["Red Sea State"],
       "startYear" : 2024,
       "startMonth" : 08,
       "startDay" : "null",
       "endYear" : 2024,
       "endMonth" : 10,
       "endDay" : "null",
       "hazards" : ["Flood"],
       "impactsAnnotation" : ["Flash floods impacted 100 to 200 kg of crop production in Red Sea State between August to October 2024"]
      },
      {
       "impactType" : "Infrastructure",
       "impactSubtype" : "Healthcare Infrastructure",
       "impactValue": "4",
       "impactUnit": "healthcare facilities",
       "impactValuePrecision" : "approx",
       "country" : "Sudan",
       "location": ["River Nile State"],
       "startYear": "null",
       "startMonth": "null",
       "startDay": "null",
       "endYear": "null",
       "endMonth": "null",
       "endDay": "null",
       "hazards" : ["Convective Storm],
       "impactsAnnotation" : ["At least 4 hospitals have been impacted by a hailstorm in River Nile State alone."]
       }]
    """
examples_range = """[{
       "impactSubtype" : "Affected People",
       "impactValue": 10000,
       "impactValueApprox" : "exact",
       "impactValueMin" : 10000,
       "impactValueMax" : 10000,
       "impactUnit": "people",
       "country" : ["Sudan"],
       "location" : ["Abu Hamad", "Tokar"],
       "startYear" :"2024,
       "startMonth" :"08,
       "startDay" : 29,
       "endYear" : "null",
       "endMonth" : "null",
       "endDay" : "null",
       "hazards" : ["Mass Movement"],
       "impactsAnnotation" : ["Landslides impacted 10000 people in the cities of Abu Hamad and Tokar on the 29 August 2024",]
      },
      {
       "impactSubtype" : "Crop Production and Forestry",
       "impactValue": 200,
       "impactValueApprox" : "approx",
       "impactValueMin" : 100,
       "impactValueMax" : 200,
       "impactUnit": "kg of crop production",
       "country" : ["Sudan"],
       "location" : ["Red Sea State"],
       "startYear" : 2024,
       "startMonth" : 08,
       "startDay" : "null",
       "endYear" : 2024,
       "endMonth" : 10,
       "endDay" : "null",
       "hazards" : ["Flood"],
       "impactsAnnotation" : ["Flash floods impacted 100 to 200 kg of crop production in Red Sea State between August to October 2024"]
      },
      {
       "impactSubtype" : "Healthcare Infrastructure",
       "impactValue": 4,
       "impactValueApprox" : "approx",
       "impactValueMin" : 4,
       "impactValueMax" : 4,
       "impactUnit": "healthcare facilities",
       "country" : ["Sudan"],
       "location": ["River Nile State"],
       "startYear": "null",
       "startMonth": "null",
       "startDay": "null",
       "endYear": "null",
       "endMonth": "null",
       "endDay": "null",
       "hazards" : ["Convective Storm],
       "impactsAnnotation" : ["At least 4 hospitals have been impacted by a hailstorm in River Nile State alone."]
       }]
    """


In [9]:
from pydantic import ValidationError
import json

def get_model_response_retry(CLIENT, MODEL, prompt, output_model):
    """Get model response structured using OpenAI API allowing for a retry"""
    try:
        completion = CLIENT.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            #functions=functions
        )
        response_content = completion.choices[0].message.content
        # Parse the response content into a list of ImpactDetail objects
        response_content = json_repair.loads(response_content)
        try:
            structured_response = output_model.model_validate(response_content)
            return structured_response.model_dump()  # Return as Python object
        except ValidationError as e:
            print("Validation Error:", e)
            #allow one retry prompting the model with its error
            prompt_system = f"""
                The previous response was not valid, leading to the following error: {e}. Please try again
                respecting the output format specified in the instructions to avoid the error."""
            try:
                completion = CLIENT.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": prompt_system},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0,
                    #functions=functions
                )
                response_content = completion.choices[0].message.content
                # Parse the response content into a list of ImpactDetail objects
                response_content = json_repair.loads(response_content)
                try:
                    structured_response = output_model.model_validate(response_content)
                    return structured_response.model_dump()  # Return as Python object
                except ValidationError as e:
                    print("Validation Error:", e)
                    return response_content
            except Exception as e:
                print("API Error:", e)
            return {"error": "Response validation failed.", "details": str(e)}
    except Exception as e:
        print("API Error:", e)
        return {"error": "API call failed.", "details": str(e)}

In [19]:
def get_event_impacts_v2(df_labelled, prompt_function, impmain, impsub, impunit, impunittype, hazards, output_model, res_savename=None):
    """Wrapper function to do all level promptings for impact extraction
    Version 2 doing a identification of all variables altogether

    """
    response = []
    response_df_list = []
    count = 0
    start_time = time.time()
    for rowid, row in df_labelled.iterrows():

        reference_info = {
            "appealCode": row["appealCode"],
            #"country_kw": row["location"],
            #"reportDate" : row["reportDate"],
            #"reportLink" : row["reportLink"],
            #"disasterType": row["disasterType"],
            "nathaz_text": row["text"]
        }
        text = row["text"]
        columns = ["appealCode", "country_kw", "reportDate", "disasterType", "impactType", "impactValue", "impactValueApprox",
                   "impactValueMin", "impactValueMax", "impactUnit", "country", "location", "startYear", "startMonth",
                   "startDay", "endYear", "endMonth", "endDay", "hazards", "impactsAnnotation"]
        data = reference_info
        #query impact, value, loc, date haz altogether
        prompt = prompt_function(text,
                                                                    imp_main=impmain,
                                                                    imp_sub=impsub,
                                                                    imp_unit=impunit,
                                                                    imp_unittype=impunittype,
                                                                    hazard_cat=hazards)
        #prompt_system = make_prompt_system(impcat, hazcat)
        #answer_impacts = get_model_response(CLIENT, MODEL_NAME, prompt, response_model=output_model)

        answer_impacts = get_model_response_retry(CLIENT, MODEL_NAME, prompt, output_model=output_model)

        #answer_impacts = json_repair.loads(result)
        #further clean-up
        #answer_impacts = list(chain(*answer_impacts)) #unlist elements
        answer_impacts = [el for el in answer_impacts if isinstance(el, dict)] #filter out anything that is not dict or list
        #answer_impacts = list(chain.from_iterable(el if isinstance(el, list) else [el] for el in answer_impacts)) #unzip list elements
        #print(f"Impacts {answer_impacts} identified in {reference_info['appealCode']}, {reference_info['reportDate']}")
        if answer_impacts:
            data = deepcopy(add_key_value_pairs(answer_impacts, data))
            response.append(data)
            #response_unnested = list(chain(*response))
            #construct df
            new_dfs = pd.concat([pd.DataFrame.from_dict(impdict, orient="index").T for impdict in data],axis=0)
        else:
            #if extraction fail, write empty row with reference info
            new_dfs = pd.DataFrame(columns=columns, data=[reference_info])

        response_df_list.append(new_dfs)
        all_response_df = pd.concat(response_df_list, ignore_index=True, axis=0)
        if res_savename:
            all_response_df.to_csv(DATA_OUT_LLMS + res_savename, index=False)

    end_time = time.time()
    nreports = len(df_labelled)
    dtime = end_time - start_time
    n_extracted_fields = len(all_response_df) if answer_impacts else 0
    #print(f"{MODEL_NAME}; time taken: {dtime} seconds, {dtime/nreports} seconds per report")
    #store time in df
    #df_time = pd.DataFrame({"model": [MODEL_NAME], "time_per_report": [dtime/nreports], "n_extracted_fields": [n_extracted_fields]})
    #df_time.to_csv(DATA_PATH + "{MODEL_NAME}_time_per_report.csv", index=False, mode="a", header=False)

    return (response, all_response_df)

In [20]:
reports_in = test_df
#ifrc_reports_df.iloc[:nreports]

#chose prompt function
prompt_function = quantify_impacts_type_value_loc_date_haz_free_unit_ranges
#chose hazard and impact cats
hazcat = hazard_main_types_emdat_extended
impmaintype = impactType_list
impsubtype = impactSubtype_list
impunit = impactUnit_list_prompting
impunittype = impactUnitType_list
constr_unit = False #constrain unit or not

#savename
sim_name = "test_range_hazmain_nb_format_std_units_units"
res_savename = f"llm_response_impact_{sim_name}_TEST_{MODEL_NAME.replace('/', '_')}.csv"
if constr_unit:
    #set up output model
    ImpactDetailConstUnit.set_allowed_classes(
                impact_types=impmaintype,
                impact_subtypes=impsubtype,
                impact_units=impunit,
                hazard_types=hazcat
            )
    output_model = ImpactListConstUnit

else:
    #set up output model
    ImpactDetailRange.set_allowed_classes(
                #impact_types=impmaintype,
                impact_subtypes=impsubtype,
                hazard_types=hazcat
            )
    output_model = ImpactListRange
print(f"Processing {res_savename}")

response, response_df = get_event_impacts_v2(reports_in, prompt_function,
                                             impmain=impmaintype,
                                             impsub=impsubtype, impunit=impunit,
                                             impunittype=impunittype,
                                             hazards=hazcat, output_model=output_model,
                                             res_savename=res_savename)

Processing llm_response_impact_test_range_hazmain_nb_format_std_units_units_TEST_meta-llama_llama-4-scout-17b-16e-instruct.csv
Validation Error: 6 validation errors for ImpactListRange
0.country
  Input should be a valid list [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.9/v/list_type
0.hazards
  Value error, Expected a list for hazards, got NoneType [type=value_error, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.9/v/value_error
1.country
  Input should be a valid list [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.9/v/list_type
1.hazards
  Value error, Expected a list for hazards, got NoneType [type=value_error, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.9/v/value_error
2.country
  Input should be a valid list [type=list_typ

In [21]:
response_df_proc = cp.deepcopy(response_df)
num_cols = ["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["location", "hazards", "impactsAnnotation"]

response_df_proc = cp.deepcopy(response_df_proc)
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)


In [22]:
response_df_proc

,impactSubtype,impactValue,impactValueApprox,impactValueMin,impactValueMax,impactUnit,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,impactsAnnotation,appealCode,nathaz_text
0,Affected People,5000.0,exact,5000.0,5000.0,people,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],"[Over 5,000 people have been impacted by block...",NaN,"Over 5,000 people have been impacted by blocke..."
1,Access to Power and Energy,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],"[Over 5,000 people have been impacted by block...",NaN,"Over 5,000 people have been impacted by blocke..."
2,Access to Healthcare,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],"[Over 5,000 people have been impacted by block...",NaN,"Over 5,000 people have been impacted by blocke..."
3,Road Infrastructure,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],"[Over 5,000 people have been impacted by block...",NaN,"Over 5,000 people have been impacted by blocke..."
4,Access to Healthcare,100.0,exact,100.0,100.0,people,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],[100 people lost access to healthcare.],NaN,100 people lost access to healthcare.
5,Road Infrastructure,100.0,exact,100.0,100.0,million USD,NaN,[],NaN,NaN,NaN,NaN,NaN,NaN,[],[More of100m USD (110m CHF) of damage to infra...,NaN,More of 100m USD (110m CHF) of damage to infra...
